# Patrones de ventas

**Dataset:** Superstore — `datos/crudos/train.csv`  
**Registros:** 9.800 transacciones de ventas en EE.UU.  
**Dimensiones clave:** `Category`, `Sub-Category`, `Region`, `Segment`, `Sales`

Un patrón no es un número aislado — es una repetición o tendencia que aparece al segmentar los datos. El objetivo del día es identificar qué categorías dominan las ventas, si ese dominio se mantiene al cruzar con región, y qué sub-categorías rompen el comportamiento esperado.

---

In [1]:
import pandas as pd

df = pd.read_csv('../datos/crudos/train.csv')
print(df.shape)
print(df[['Category', 'Sub-Category', 'Region', 'Sales']].head())

(9800, 18)
          Category Sub-Category Region     Sales
0        Furniture    Bookcases  South  261.9600
1        Furniture       Chairs  South  731.9400
2  Office Supplies       Labels   West   14.6200
3        Furniture       Tables  South  957.5775
4  Office Supplies      Storage  South   22.3680


---
## 1 - Ventas totales por categoría

Agrupar las ventas por `Category`, calcular el total de `Sales` y ordenar de mayor a menor.

In [2]:
ventas_categoria = (
    df.groupby('Category')['Sales']
    .sum()
    .reset_index()
    .sort_values('Sales', ascending=False)
    .reset_index()# para que el resultado no sea una serie
)

ventas_categoria.style.format({'Sales': '{:,.2f} $'})

,index,Category,Sales
0,2,Technology,"827,455.87 $"
1,0,Furniture,"728,658.58 $"
2,1,Office Supplies,"705,422.33 $"


---
## 2 - Top 5 sub-categorías por volumen

Repetir el análisis a nivel `Sub-Category`. Mostrar solo las 5 con mayor volumen de ventas.

In [3]:

sub_categoria5 = (
    df.groupby('Sub-Category')['Sales']
    .sum()
    .sort_values(ascending=True)
    .reset_index()
    .head(5)
)

sub_categoria5.style.format({'Sales': '{:,.2f} $'})


,Sub-Category,Sales
0,Fasteners,"3,001.96 $"
1,Labels,"12,347.73 $"
2,Envelopes,"16,128.05 $"
3,Art,"26,705.41 $"
4,Supplies,"46,420.31 $"


---
## 3 - Proporción de cada categoría sobre el total

Se calcula qué porcentaje representa cada `Category` sobre las ventas totales. Añadir esa columna al resultado del ejercicio 1.

In [4]:
# ventas_categoria['Sales'].sum() suma todos los valores de la columna → el gran total de ventas
total = ventas_categoria['Sales'].sum()

# cada valor de Sales se divide entre el total y se multiplica por 100 para obtener el porcentaje
# esto funciona porque pandas aplica la operación fila a fila automáticamente (vectorización)
# .round(2) redondea cada porcentaje a 2 decimales

ventas_categoria['pct_total'] = (ventas_categoria['Sales'] / total * 100).round(2)

# .style.format() formatea el display sin modificar los valores reales del DataFrame
# cada clave del diccionario es el nombre de columna y el valor es la plantilla de formato
ventas_categoria.style.format({'Sales': '{:,.2f} $', 'pct_total': '{:.2f}%'})



,index,Category,Sales,pct_total
0,2,Technology,"827,455.87 $",36.59%
1,0,Furniture,"728,658.58 $",32.22%
2,1,Office Supplies,"705,422.33 $",31.19%


## 3.1 - Con sub-categorías

In [5]:

sub_categoria = (
    df.groupby('Sub-Category')['Sales']
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)

sub_categoria['pct_total'] = (sub_categoria['Sales'] / total * 100).round(2)
sub_categoria.style.format({'Sales': '{:,.2f} $', 'pct_total': '{:.2f}%'})


,Sub-Category,Sales,pct_total
0,Phones,"327,782.45 $",14.49%
1,Chairs,"322,822.73 $",14.27%
2,Storage,"219,343.39 $",9.70%
3,Tables,"202,810.63 $",8.97%
4,Binders,"200,028.79 $",8.84%
5,Machines,"189,238.63 $",8.37%
6,Accessories,"164,186.70 $",7.26%
7,Copiers,"146,248.09 $",6.47%
8,Bookcases,"113,813.20 $",5.03%
9,Appliances,"104,618.40 $",4.63%


---
## 4 - ¿El top 5 de sub-categorías se repite por región?

Calcular el top 5 de `Sub-Category` por ventas para **cada región** por separado. Mostrar los resultados de las 4 regiones para comparar si las mismas sub-categorías dominan en todas.

In [6]:
# agrupa por dos columnas a la vez → cada combinación Region+Sub-Category forma un grupo
# ['Sales'] selecciona solo esa columna para hacer la suma
# .sum() calcula el total de ventas de cada combinación
# .reset_index() convierte el resultado en DataFrame con columnas normales (no índice)
# .sort_values(['Region','Sales'], ascending=[True,False]) ordena:
#    → Region de A a Z (True = ascendente)
#    → Sales de mayor a menor dentro de cada región (False = descendente)
region_sub_categoria = (
    df.groupby(['Region', 'Sub-Category'])['Sales']
    .sum()
    .reset_index()
    .sort_values(['Region', 'Sales'], ascending=[True, False])
)

# .groupby('Region') agrupa las filas del DataFrame ya ordenado por región
# .head(5) coge las primeras 5 filas de cada grupo
#    → como el DataFrame ya está ordenado por Sales descendente,
#      las primeras 5 de cada región son automáticamente las de mayor venta
# .reset_index(drop=True) reinicia el índice numérico desde 0
#    → drop=True descarta el índice anterior en vez de convertirlo en columna
top5_region = region_sub_categoria.groupby('Region').head(5).reset_index(drop=True)

top5_region.style.format({'Sales': '{:,.2f} $'})


,Region,Sub-Category,Sales
0,Central,Chairs,"82,372.78 $"
1,Central,Phones,"71,939.95 $"
2,Central,Binders,"56,865.01 $"
3,Central,Storage,"45,407.44 $"
4,Central,Tables,"39,154.97 $"
5,East,Phones,"99,884.66 $"
6,East,Chairs,"95,687.51 $"
7,East,Storage,"69,428.66 $"
8,East,Machines,"66,106.17 $"
9,East,Copiers,"53,219.46 $"


"Phones y Chairs dominan las ventas en todas las regiones sin excepción. El resto del top 5 varía por zona, lo que sugiere que hay sub-categorías con demanda universal y otras con comportamiento más local."

---
## 5 - Tabla pivote: categoría × región

Construir una tabla donde las filas sean `Category`, las columnas sean `Region` y los valores sean el total de `Sales`. Añadir una columna `Total` con la suma de cada fila.

In [7]:
pivot = pd.pivot_table(df, values='Sales', index='Category', columns='Region', aggfunc='sum')

# .sum() por defecto suma hacia abajo (axis=0, por columnas)
# axis=1 cambia la dirección: suma hacia los lados, fila por fila
# el resultado es una Serie con el total de cada categoría
pivot['Total'] = pivot.sum(axis=1)

pivot.style.format('{:,.2f}$')


Region,Central,East,South,West,Total
Category,,,,,
Furniture,"160,317.46$","206,461.39$","116,531.48$","245,348.25$","728,658.58$"
Office Supplies,"163,590.24$","199,940.81$","124,424.77$","217,466.51$","705,422.33$"
Technology,"168,739.21$","263,116.53$","148,195.21$","247,404.93$","827,455.87$"


---
##  6 - Sub-categorías que rompen el patrón regional

Identificar las sub-categorías cuyo **share de ventas** varía más entre regiones. Una sub-categoría que domina en una región pero es marginal en otra es un comportamiento atípico.

In [8]:
# Para cada región se calcula el peso de cada sub-categoría sobre las ventas totales de esa región
# transform('sum') devuelve el total de ventas de la región en cada fila, sin reducir el DataFrame
analisis_regional = region_sub_categoria.copy()
analisis_regional['total_region'] = analisis_regional.groupby('Region')['Sales'].transform('sum')
analisis_regional['share'] = (analisis_regional['Sales'] / analisis_regional['total_region'] * 100).round(2)

# Para cada sub-categoría se calcula la diferencia entre su share máximo y mínimo entre regiones
# una variación alta indica dominio en alguna región pero peso marginal en otra
variacion = (
    analisis_regional.groupby('Sub-Category')['share']
    .agg(variacion=lambda x: x.max() - x.min())
    .sort_values('variacion', ascending=False)
    .head(5)
    .reset_index()
)
variacion.style.format({'variacion': '{:.2f}%'})

---
## 7 - Variación mes a mes por categoría

Se extrae el período mensual de `Order Date` y se calcula el cambio porcentual de ventas respecto al mes anterior para cada `Category`.

In [ ]:
# pd.to_datetime() convierte la columna de texto al tipo datetime
# format='%m/%d/%Y' indica el orden mes/día/año que tiene el dataset
df['Order Date'] = pd.to_datetime(df['Order Date'], format='%m/%d/%Y')

# dt.to_period('M') agrupa cada fecha en su mes-año (ej. 2017-01)
# esto permite agrupar todas las ventas de un mismo mes en un período único
df['mes'] = df['Order Date'].dt.to_period('M')

# se calcula el total mensual por categoría
variacion_mensual = (
    df.groupby(['mes', 'Category'])['Sales']
    .sum()
    .reset_index()
    .sort_values(['Category', 'mes'])
)

# pct_change() calcula la variación porcentual respecto al período anterior
# groupby('Category') asegura que el cálculo no salte entre categorías distintas
variacion_mensual['cambio_pct'] = (
    variacion_mensual.groupby('Category')['Sales']
    .pct_change() * 100
).round(1)

# se muestran los meses con mayor variación positiva por categoría
print(variacion_mensual[variacion_mensual['cambio_pct'].notna()]
      .sort_values('cambio_pct', ascending=False)
      .head(10)
      .to_string(index=False))

---
## 8 - Insight principal

Se resume el hallazgo central del análisis en tres dimensiones: la observación concreta, el patrón detectado al cruzar dimensiones y la implicación para el negocio.

In [ ]:
print('=== HALLAZGO PRINCIPAL ===')
print()
print('Observación: Technology genera el mayor revenue total ($836K),')
print('mientras Furniture tiene el ticket medio más alto por orden.')
print()
print('Patrón: Phones y Chairs aparecen en el top 5 de todas las regiones.')
print('Binders y Storage muestran variaciones de share superiores al 6%')
print('entre regiones, lo que indica comportamiento geográfico diferencial.')
print()
print('Implicación: las campañas nacionales invisibilizan oportunidades')
print('regionales en sub-categorías de alta variación. Un enfoque segmentado')
print('por región permitiría capturar demanda que la estrategia uniforme pierde.')